# n2_train_text_conditional_diffusion.ipynb

This notebook trains a **text-conditioned diffusion model** on 50,000 RGBA pixel-art character sprites (128x128, 4-view format) using **optimized textual captions**.  
Compared to the unconditional baseline in *n1*, this model incorporates caption embeddings as conditioning signals, enabling controllable sprite generation guided by structured textual descriptions.

The model serves as the **text-to-image diffusion baseline** for the PIXEL-T2I project, allowing direct comparison between unconditional and text-conditioned generation under identical architectural and training settings.  
Training takes approximately **7 hours on an A100 GPU**, and model checkpoints are saved to Google Drive for later inference and evaluation.

## Section 1. Setup and Imports

In [1]:
# Install required packages
!pip install -q accelerate transformers

# Standard library
import os
import math
import time
import warnings
from pathlib import Path
from typing import List
import csv
from datetime import datetime

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils as nn_utils
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler

# PyTorch optimization
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# Vision / Image processing
from torchvision.utils import save_image, make_grid
from PIL import Image
import torchvision.transforms as T

# Visualization
import matplotlib.pyplot as plt

# Progress bar
from tqdm.auto import tqdm

# CLIP for text encoding
from transformers import CLIPTokenizer, CLIPTextModel

print("All libraries imported successfully")

All libraries imported successfully


## Section 2. Environment & Device Setup

In [2]:
# 2.1 Environment check
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2.2 Mount Google Drive and prepare dataset
from google.colab import drive
import shutil
import zipfile

drive.mount('/content/drive')

# Drive paths
DRIVE_ROOT = Path("/content/drive/MyDrive/PIXEL-T2I").resolve()
DRIVE_ZIP_PATH = DRIVE_ROOT / "pixel_character_dataset" / "processed" / "dataset_4view.zip"

# Local SSD paths (fast reading)
LOCAL_DATA = Path("/content/data_cache")
LOCAL_DATASET = LOCAL_DATA / "dataset_4view"
IMAGE_DIR = LOCAL_DATASET / "images"
CAPTION_CSV = LOCAL_DATASET / "captions.csv"
CAPTION_OPT_CSV = LOCAL_DATASET / "captions_optimized.csv"

# Copy dataset to local SSD if not exists
if not IMAGE_DIR.exists():
    print(f"Copying dataset from Drive...")

    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    local_zip_path = Path("/content/temp_dataset.zip")

    shutil.copy(DRIVE_ZIP_PATH, local_zip_path)

    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA)

    os.remove(local_zip_path)
    print("Dataset copied to local SSD")
else:
    print("Dataset already in local SSD")

# Verify dataset
if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Cannot find images at {IMAGE_DIR}")

num_images = len(list(IMAGE_DIR.glob("*.png")))
print(f"Dataset ready: {num_images} images at {IMAGE_DIR}")

# Verify captions
if not CAPTION_CSV.exists():
    raise FileNotFoundError(f"Cannot find captions.csv at {CAPTION_CSV}")
if not CAPTION_OPT_CSV.exists():
    raise FileNotFoundError(f"Cannot find captions_optimized.csv at {CAPTION_OPT_CSV}")

print(f"Captions ready: {CAPTION_CSV.name}, {CAPTION_OPT_CSV.name}")

# 2.3 Set paths
ROOT = DRIVE_ROOT
DATASET_ROOT = LOCAL_DATASET

print("\nSection 2 complete")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Mounted at /content/drive
Copying dataset from Drive...
Dataset copied to local SSD
Dataset ready: 50000 images at /content/data_cache/dataset_4view/images
Captions ready: captions.csv, captions_optimized.csv

Section 2 complete


## Section 3. Configuration (text-conditional)

In [3]:
CONFIG = {
    # Data paths
    "image_dir": IMAGE_DIR,
    "caption_csv": CAPTION_OPT_CSV,

    # Model and output paths - Google Drive
    "checkpoint_dir": ROOT / "models" / "pixel_text_conditional" / "checkpoints",
    "final_model_dir": ROOT / "models" / "pixel_text_conditional",
    "log_dir": ROOT / "outputs" / "logs_text_conditional",
    "sample_dir": ROOT / "outputs" / "samples_text_conditional",

    # Image settings
    "image_size": 128,
    "in_channels": 4,  # RGBA format

    # DataLoader settings
    "batch_size": 48,
    "num_workers": 2,
    "shuffle": True,
    "pin_memory": True,

    # Diffusion parameters
    "timesteps": 1000,              # Total diffusion steps
    "beta_schedule": "linear",      # "linear" or "cosine"
    "beta_start": 0.0001,           # Starting beta value
    "beta_end": 0.02,               # Ending beta value

    # Text conditioning
    "clip_model_name": "openai/clip-vit-base-patch32",
    "text_embed_dim": 512,           # CLIP output dimension
    "max_text_length": 77,           # CLIP max token length
    "cfg_drop_prob": 0.1,            # Classifier-free guidance drop probability
    "cfg_scale": 1.0,                # Guidance scale for inference

    # UNet architecture
    "unet_channels": 128,              # Base channel count (increased since no text conditioning)
    "channel_mult": (1, 2, 4, 4),      # Channel multipliers per resolution
    "num_res_blocks": 2,               # Residual blocks per resolution
    "attention_resolutions": (8, 16),  # Resolutions to apply self-attention
    "dropout": 0.1,                    # Dropout probability (can increase for unconditional)
    "num_heads": 8,                    # Number of attention heads
    "use_cross_attention": True,       # Enable cross-attention

    # Training settings
    "num_epochs": 30,
    "learning_rate": 1e-4,
    "weight_decay": 0.01,
    "lr_scheduler": "cosine",          # "cosine" or "constant"
    "warmup_steps": 50,
    "gradient_clip": 1.0,
    "use_mixed_precision": True,       # Enable FP16 training

    # Training control
    "start_training": True,      # Set to True to start fresh training
    "resume_training": False,    # Set to True to resume from checkpoint

    # Logging and checkpointing
    "log_every": 50,                   # Log every N batches
    "save_every_epochs": 5,            # Save checkpoint every N epochs
    "keep_last_n": 3,                  # Keep only last N checkpoints
    "sample_every_epochs": 5,          # Generate samples every N epochs (more frequent)
    "num_samples": 4,                  # Number of samples to generate

    # Sampling settings
    "default_ddim_steps": 10,

    # Testing and debugging
    "quick_test": False,                # Enable quick test mode (subset of data)
    "quick_test_samples": 100,          # Number of samples for quick test
}

print("Configuration loaded successfully")
print(f"\nKey settings:")
print(f"  Dataset images: {CONFIG['image_dir']}")
print(f"  Captions CSV:   {CONFIG['caption_csv']}")
print(f"  CLIP model:     {CONFIG['clip_model_name']}")
print(f"  Text embed dim: {CONFIG['text_embed_dim']}")
print(f"  Checkpoints:    {CONFIG['checkpoint_dir']}")
print(f"  Logs:           {CONFIG['log_dir']}")
print(f"  Samples:        {CONFIG['sample_dir']}")
print(f"  Batch size:     {CONFIG['batch_size']}")
print(f"  Epochs:         {CONFIG['num_epochs']}")
print(f"  Quick test:     {CONFIG['quick_test']}")

print("\nSection 3 complete")

Configuration loaded successfully

Key settings:
  Dataset images: /content/data_cache/dataset_4view/images
  Captions CSV:   /content/data_cache/dataset_4view/captions_optimized.csv
  CLIP model:     openai/clip-vit-base-patch32
  Text embed dim: 512
  Checkpoints:    /content/drive/MyDrive/PIXEL-T2I/models/pixel_text_conditional/checkpoints
  Logs:           /content/drive/MyDrive/PIXEL-T2I/outputs/logs_text_conditional
  Samples:        /content/drive/MyDrive/PIXEL-T2I/outputs/samples_text_conditional
  Batch size:     48
  Epochs:         30
  Quick test:     False

Section 3 complete


## Section 4. Dataset & DataLoader

In [4]:
# 4.1 Dataset for 4-view sprites (Text Conditional)
class Sprite4ViewDataset(Dataset):
    """
    Text-conditional sprite dataset with RGBA images and captions.
    """
    def __init__(self, config):
        self.config = config
        self.image_dir = Path(config["image_dir"])
        self.image_size = config["image_size"]

        # Load image filenames
        self.image_files = sorted([f for f in os.listdir(self.image_dir) if f.endswith('.png')])

        # Load captions
        self.captions = self._load_captions(config["caption_csv"])

        print(f"Loaded {len(self.image_files)} images from {self.image_dir}")
        print(f"Loaded {len(self.captions)} captions from {config['caption_csv']}")

        # Define transforms
        self.transform = self._get_transforms()

    def _load_captions(self, caption_csv_path):
        """
        Load captions from CSV file.

        Returns:
            dict: {filename: caption}
        """
        captions = {}

        with open(caption_csv_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                filename = row['image_path']
                caption = row['text']
                captions[filename] = caption

        return captions

    def _get_transforms(self):
        """Define image transformations for RGBA sprites"""
        return T.Compose([
            T.Lambda(lambda img: img.convert('RGBA') if img.mode != 'RGBA' else img),
            T.Resize((self.image_size, self.image_size), interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor(),
            T.Normalize(mean=[0.5]*4, std=[0.5]*4),
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        """
        Returns:
            dict: {
                'image': tensor (4, H, W),
                'caption': str,
                'filename': str
            }
        """
        img_name = self.image_files[idx]
        img_path = self.image_dir / img_name

        # Load image
        try:
            image = Image.open(img_path)
            image = self.transform(image)
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            image = torch.zeros(4, self.image_size, self.image_size)

        # Get caption
        caption = self.captions.get(img_name, "")  # Empty string if not found

        if not caption:
            print(f"Warning: No caption for {img_name}")

        return {
            'image': image,
            'caption': caption,
            'filename': img_name
        }


# 4.2 DataLoader Creation
def create_dataloader(config):
    """Create training dataloader."""
    dataset = Sprite4ViewDataset(config)

    # Quick test mode
    if config.get("quick_test", False):
        n_samples = config.get("quick_test_samples", 100)
        dataset = Subset(dataset, range(min(n_samples, len(dataset))))
        print(f"Quick test mode: using {len(dataset)} samples")

    # Create dataloader
    train_loader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=config["shuffle"],
        num_workers=config["num_workers"],
        pin_memory=config["pin_memory"],
        drop_last=True
    )

    print(f"DataLoader created: {len(dataset)} samples, {len(train_loader)} batches")

    return train_loader


# 4.3 Create DataLoader
train_loader = create_dataloader(CONFIG)

# 4.4 Test DataLoader (optional)
print("\nTesting DataLoader...")
test_batch = next(iter(train_loader))
print(f"  Image batch shape: {test_batch['image'].shape}")
print(f"  Number of captions: {len(test_batch['caption'])}")
print(f"  Sample caption: '{test_batch['caption'][0]}'")

print("\nSection 4 complete")

Loaded 50000 images from /content/data_cache/dataset_4view/images
Loaded 50000 captions from /content/data_cache/dataset_4view/captions_optimized.csv
DataLoader created: 50000 samples, 1041 batches

Testing DataLoader...
  Image batch shape: torch.Size([48, 4, 128, 128])
  Number of captions: 48
  Sample caption: 'lpc-style pixel art character, with male human body, dark skin, wearing teal tunic, magenta pants, white cloth bracers and brown slippers, with brown long mohawk hairstyle, with spear, 4-view sprite sheet (front/back/left/right), sharp small pixel art, hard edges, no anti-aliasing'

Section 4 complete


## Section 5. Diffusion Process

In [5]:
# 5.1 Noise Schedule Functions
def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
    """Linear schedule from DDPM paper."""
    return torch.linspace(beta_start, beta_end, timesteps)


def cosine_beta_schedule(timesteps, s=0.008):
    """Cosine schedule from Improved DDPM paper."""
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)


def get_beta_schedule(schedule_name, timesteps, beta_start=0.0001, beta_end=0.02):
    """Get beta schedule based on name."""
    if schedule_name == "linear":
        return linear_beta_schedule(timesteps, beta_start, beta_end)
    elif schedule_name == "cosine":
        return cosine_beta_schedule(timesteps)
    else:
        raise ValueError(f"Unknown beta schedule: {schedule_name}")


# 5.2 Precompute diffusion parameters
def prepare_noise_schedule(config):
    """Precompute all noise schedule parameters."""
    timesteps = config["timesteps"]

    betas = get_beta_schedule(
        config["beta_schedule"],
        timesteps,
        config["beta_start"],
        config["beta_end"]
    )

    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
    sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
    posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

    noise_schedule = {
        "betas": betas,
        "alphas": alphas,
        "alphas_cumprod": alphas_cumprod,
        "alphas_cumprod_prev": alphas_cumprod_prev,
        "sqrt_alphas_cumprod": sqrt_alphas_cumprod,
        "sqrt_one_minus_alphas_cumprod": sqrt_one_minus_alphas_cumprod,
        "sqrt_recip_alphas": sqrt_recip_alphas,
        "posterior_variance": posterior_variance,
        'timesteps': timesteps,
    }

    return noise_schedule


# 5.3 Forward Diffusion Process
def extract(a, t, x_shape):
    """Extract coefficients at timesteps and reshape for broadcasting."""
    batch_size = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))


def q_sample(x_start, t, noise, noise_schedule):
    """
    Forward diffusion: q(x_t | x_0).
    Formula: x_t = sqrt(alpha_cumprod_t) * x_0 + sqrt(1 - alpha_cumprod_t) * noise
    """
    sqrt_alphas_cumprod_t = extract(noise_schedule["sqrt_alphas_cumprod"], t, x_start.shape)
    sqrt_one_minus_alphas_cumprod_t = extract(noise_schedule["sqrt_one_minus_alphas_cumprod"], t, x_start.shape)

    return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise


# 5.4 Create noise schedule
noise_schedule = prepare_noise_schedule(CONFIG)

# Move to device
for key in noise_schedule:
    if isinstance(noise_schedule[key], torch.Tensor):
        noise_schedule[key] = noise_schedule[key].to(device)

print(f"Noise schedule ready: {CONFIG['beta_schedule']}, {CONFIG['timesteps']} steps")
print("\nSection 5 complete")

Noise schedule ready: linear, 1000 steps

Section 5 complete


## Section 6. UNet Model

In [6]:
# 6.1 Building Blocks
class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal time embeddings for timestep t."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class CrossAttentionBlock(nn.Module):
    """
    Cross-attention block for text conditioning.
    Q from image features, K/V from text embeddings.
    """
    def __init__(self, channels, context_dim, num_heads=8):
        super().__init__()
        self.channels = channels
        self.context_dim = context_dim
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        assert channels % num_heads == 0, "channels must be divisible by num_heads"

        self.norm = nn.GroupNorm(32, channels)
        self.context_norm = nn.LayerNorm(context_dim)

        # Q from image, K/V from text
        self.to_q = nn.Conv2d(channels, channels, kernel_size=1)
        self.to_k = nn.Linear(context_dim, channels)
        self.to_v = nn.Linear(context_dim, channels)
        self.to_out = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x, context):
        """
        Args:
            x: Image features (B, C, H, W)
            context: Text embeddings (B, seq_len, context_dim)
        Returns:
            out: (B, C, H, W)
        """
        B, C, H, W = x.shape
        residual = x

        # Normalize
        x = self.norm(x)
        context = self.context_norm(context)

        # Q from image
        q = self.to_q(x)  # (B, C, H, W)
        q = q.reshape(B, self.num_heads, self.head_dim, H * W)
        q = q.permute(0, 1, 3, 2)  # (B, num_heads, H*W, head_dim)

        # K, V from text
        k = self.to_k(context)  # (B, seq_len, C)
        v = self.to_v(context)  # (B, seq_len, C)
        k = k.reshape(B, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        v = v.reshape(B, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        # Attention
        scale = self.head_dim ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)

        # Apply attention
        out = torch.matmul(attn, v)  # (B, num_heads, H*W, head_dim)
        out = out.permute(0, 1, 3, 2).reshape(B, C, H, W)

        # Project and residual
        out = self.to_out(out)
        return out + residual


class ResidualBlock(nn.Module):
    """
    Residual block with time embedding and optional cross-attention.
    """
    def __init__(self, in_channels, out_channels, time_emb_dim,
                 dropout=0.0, use_cross_attn=False, context_dim=None):
        super().__init__()

        self.use_cross_attn = use_cross_attn

        self.norm1 = nn.GroupNorm(32, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        self.time_emb_proj = nn.Linear(time_emb_dim, out_channels)

        self.norm2 = nn.GroupNorm(32, out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        if in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.residual_conv = nn.Identity()

        # Cross-attention
        if use_cross_attn:
            assert context_dim is not None, "context_dim must be provided for cross-attention"
            self.cross_attn = CrossAttentionBlock(out_channels, context_dim)
        else:
            self.cross_attn = None

        self.act = nn.SiLU()

    def forward(self, x, time_emb, context=None):
        """
        Args:
            x: Image features (B, C, H, W)
            time_emb: Time embeddings (B, time_emb_dim)
            context: Text embeddings (B, seq_len, context_dim) or None
        Returns:
            out: (B, C, H, W)
        """
        residual = self.residual_conv(x)

        h = self.norm1(x)
        h = self.act(h)
        h = self.conv1(h)

        time_emb = self.act(time_emb)
        time_emb = self.time_emb_proj(time_emb)
        h = h + time_emb[:, :, None, None]

        h = self.norm2(h)
        h = self.act(h)
        h = self.dropout(h)
        h = self.conv2(h)

        # Cross-attention with text
        if self.cross_attn is not None and context is not None:
            h = self.cross_attn(h, context)

        return h + residual


class AttentionBlock(nn.Module):
    """Self-attention block for spatial features."""
    def __init__(self, channels, num_heads=8):
        super().__init__()
        self.channels = channels
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        assert channels % num_heads == 0

        self.norm = nn.GroupNorm(32, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape
        residual = x

        x = self.norm(x)

        qkv = self.qkv(x)
        qkv = qkv.reshape(B, 3, self.num_heads, self.head_dim, H * W)
        qkv = qkv.permute(1, 0, 2, 4, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]

        scale = self.head_dim ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)

        out = torch.matmul(attn, v)
        out = out.permute(0, 1, 3, 2).reshape(B, C, H, W)

        out = self.proj(out)
        return out + residual


class Downsample(nn.Module):
    """Spatial downsampling by factor of 2."""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    """Spatial upsampling by factor of 2."""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        return self.conv(x)


# 6.2 Main UNet Model (Text-Conditional)
class UNet(nn.Module):
    """
    UNet for text-conditional diffusion models.
    Optimized for 128×128 RGBA sprite generation with CLIP text conditioning.
    """
    def __init__(self, config):
        super().__init__()

        self.config = config
        in_channels = config["in_channels"]
        model_channels = config["unet_channels"]
        channel_mult = config["channel_mult"]
        num_res_blocks = config["num_res_blocks"]
        attention_resolutions = config["attention_resolutions"]
        dropout = config["dropout"]
        num_heads = config["num_heads"]

        # Text conditioning
        use_cross_attn = config.get("use_cross_attention", True)
        context_dim = config.get("text_embed_dim", 512)

        time_emb_dim = model_channels * 4

        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(model_channels),
            nn.Linear(model_channels, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        # Initial conv
        self.conv_in = nn.Conv2d(in_channels, model_channels, kernel_size=3, padding=1)

        # Encoder
        self.encoder_blocks = nn.ModuleList()
        current_channels = model_channels
        current_resolution = config["image_size"]
        self.encoder_out_channels = [model_channels]

        for level, mult in enumerate(channel_mult):
            out_channels = model_channels * mult

            for i in range(num_res_blocks):
                block_layers = nn.ModuleList()

                # ResBlock with cross-attention
                block_layers.append(
                    ResidualBlock(
                        current_channels,
                        out_channels,
                        time_emb_dim,
                        dropout,
                        use_cross_attn=use_cross_attn,
                        context_dim=context_dim
                    )
                )
                current_channels = out_channels

                # Self-attention
                if current_resolution in attention_resolutions:
                    block_layers.append(AttentionBlock(current_channels, num_heads))

                self.encoder_blocks.append(block_layers)
                self.encoder_out_channels.append(current_channels)

            # Downsample
            if level != len(channel_mult) - 1:
                self.encoder_blocks.append(nn.ModuleList([Downsample(current_channels)]))
                current_resolution //= 2
                self.encoder_out_channels.append(current_channels)

        # Bottleneck
        self.mid_block1 = ResidualBlock(
            current_channels,
            current_channels,
            time_emb_dim,
            dropout,
            use_cross_attn=use_cross_attn,
            context_dim=context_dim
        )
        self.mid_attn = AttentionBlock(current_channels, num_heads)
        self.mid_block2 = ResidualBlock(
            current_channels,
            current_channels,
            time_emb_dim,
            dropout,
            use_cross_attn=use_cross_attn,
            context_dim=context_dim
        )

        # Decoder
        self.decoder_blocks = nn.ModuleList()

        for level, mult in reversed(list(enumerate(channel_mult))):
            out_channels = model_channels * mult

            for i in range(num_res_blocks + 1):
                skip_channels = self.encoder_out_channels.pop()
                block_layers = nn.ModuleList()

                # ResBlock with cross-attention
                block_layers.append(
                    ResidualBlock(
                        current_channels + skip_channels,
                        out_channels,
                        time_emb_dim,
                        dropout,
                        use_cross_attn=use_cross_attn,
                        context_dim=context_dim
                    )
                )
                current_channels = out_channels

                # Self-attention
                if current_resolution in attention_resolutions:
                    block_layers.append(AttentionBlock(current_channels, num_heads))

                self.decoder_blocks.append(block_layers)

                # Upsample
                if level != 0 and i == num_res_blocks:
                    self.decoder_blocks.append(nn.ModuleList([Upsample(current_channels)]))
                    current_resolution *= 2

        # Output
        self.conv_out = nn.Sequential(
            nn.GroupNorm(32, current_channels),
            nn.SiLU(),
            nn.Conv2d(current_channels, in_channels, kernel_size=3, padding=1),
        )

    def forward(self, x, t, context=None):
        """
        Args:
            x: Noisy images (B, 4, 128, 128)
            t: Timesteps (B,)
            context: Text embeddings (B, seq_len, text_embed_dim) or None
        Returns:
            Predicted noise (B, 4, 128, 128)
        """
        # Time embedding
        time_emb = self.time_mlp(t)

        # Initial conv
        h = self.conv_in(x)

        # Encoder
        encoder_features = [h]
        for block_layers in self.encoder_blocks:
            for layer in block_layers:
                if isinstance(layer, ResidualBlock):
                    h = layer(h, time_emb, context)  # Pass context
                elif isinstance(layer, AttentionBlock):
                    h = layer(h)
                elif isinstance(layer, Downsample):
                    h = layer(h)
            encoder_features.append(h)

        # Bottleneck
        h = self.mid_block1(h, time_emb, context)  # Pass context
        h = self.mid_attn(h)
        h = self.mid_block2(h, time_emb, context)  # Pass context

        # Decoder
        for block_layers in self.decoder_blocks:
            if any(isinstance(layer, ResidualBlock) for layer in block_layers):
                skip = encoder_features.pop()
                h = torch.cat([h, skip], dim=1)

            for layer in block_layers:
                if isinstance(layer, ResidualBlock):
                    h = layer(h, time_emb, context)  # Pass context
                elif isinstance(layer, AttentionBlock):
                    h = layer(h)
                elif isinstance(layer, Upsample):
                    h = layer(h)

        # Output
        return self.conv_out(h)


# 6.3 Weight Initialization
def init_weights(m):
    """Proper weight initialization for stable training."""
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.GroupNorm):
        if m.weight is not None:
            nn.init.ones_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


# 6.4 Create Model
unet = UNet(CONFIG)
unet.apply(init_weights)
unet = unet.to(device)

total_params = sum(p.numel() for p in unet.parameters())
trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)

print(f"UNet created: {total_params / 1e6:.2f}M parameters")
print(f"  Trainable: {trainable_params / 1e6:.2f}M")
print(f"  Cross-attention: {'Enabled' if CONFIG.get('use_cross_attention') else 'Disabled'}")
print(f"  Text embed dim: {CONFIG.get('text_embed_dim', 512)}")
print("\nSection 6 complete")

UNet created: 116.56M parameters
  Trainable: 116.56M
  Cross-attention: Enabled
  Text embed dim: 512

Section 6 complete


## Section 7. Training Setup

In [7]:
# 7.1 Load CLIP Text Encoder
# Load tokenizer and text encoder
tokenizer = CLIPTokenizer.from_pretrained(CONFIG["clip_model_name"])
text_encoder = CLIPTextModel.from_pretrained(CONFIG["clip_model_name"])

# Move to device and freeze
text_encoder = text_encoder.to(device)
text_encoder.eval()

# Freeze CLIP weights (we don't train CLIP)
for param in text_encoder.parameters():
    param.requires_grad = False

print(f"CLIP loaded: {CONFIG['clip_model_name']}")
print(f"  Text encoder params: {sum(p.numel() for p in text_encoder.parameters()) / 1e6:.2f}M (frozen)")
print(f"  Max text length: {CONFIG['max_text_length']}")
print(f"  Text embed dim: {CONFIG['text_embed_dim']}")


# 7.2 Optimizer and Scheduler
# Optimizer (only UNet parameters, CLIP is frozen)
optimizer = torch.optim.AdamW(
    unet.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    betas=(0.9, 0.999)
)
print(f"\nOptimizer: AdamW (lr={CONFIG['learning_rate']})")

# Learning rate scheduler
total_steps = len(train_loader) * CONFIG["num_epochs"]
warmup_steps = min(CONFIG["warmup_steps"], total_steps // 2)

if CONFIG["lr_scheduler"] == "cosine" and total_steps > warmup_steps:
    # Warmup scheduler
    warmup_scheduler = LinearLR(
        optimizer,
        start_factor=0.01,
        end_factor=1.0,
        total_iters=warmup_steps
    )

    # Cosine scheduler
    cosine_steps = total_steps - warmup_steps
    cosine_scheduler = CosineAnnealingLR(
        optimizer,
        T_max=max(1, cosine_steps),
        eta_min=CONFIG["learning_rate"] * 0.1
    )

    # Combine warmup + cosine
    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_steps]
    )

    print(f"Scheduler: Warmup + CosineAnnealing (warmup={warmup_steps}, total={total_steps})")
else:
    scheduler = None
    print(f"Scheduler: None")

# Mixed precision
if CONFIG["use_mixed_precision"]:
    scaler = GradScaler()
    print(f"Mixed precision: Enabled")
else:
    scaler = None
    print(f"Mixed precision: Disabled")


# 7.3 Create Directories
checkpoint_dir = Path(CONFIG["checkpoint_dir"])
checkpoint_dir.mkdir(parents=True, exist_ok=True)

log_dir = Path(CONFIG["log_dir"])
log_dir.mkdir(parents=True, exist_ok=True)

sample_dir = Path(CONFIG["sample_dir"])
sample_dir.mkdir(parents=True, exist_ok=True)

print(f"\nDirectories ready:")
print(f"  Checkpoints: {checkpoint_dir}")
print(f"  Logs: {log_dir}")
print(f"  Samples: {sample_dir}")

print("\nSection 7 complete")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP loaded: openai/clip-vit-base-patch32
  Text encoder params: 63.17M (frozen)
  Max text length: 77
  Text embed dim: 512

Optimizer: AdamW (lr=0.0001)
Scheduler: Warmup + CosineAnnealing (warmup=50, total=31230)
Mixed precision: Enabled


/tmp/ipython-input-398109130.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]


Directories ready:
  Checkpoints: /content/drive/MyDrive/PIXEL-T2I/models/pixel_text_conditional/checkpoints
  Logs: /content/drive/MyDrive/PIXEL-T2I/outputs/logs_text_conditional
  Samples: /content/drive/MyDrive/PIXEL-T2I/outputs/samples_text_conditional

Section 7 complete


## Section 8. DDIM Sampling

In [8]:
# 8.1 DDIM Sampling Functions
@torch.no_grad()
def ddim_sample_step(unet, x, t, t_prev, noise_schedule, context=None, eta=0.0):
    """
    Single DDIM sampling step.

    Args:
        unet: UNet model
        x: Current noisy image (B, C, H, W)
        t: Current timestep (B,)
        t_prev: Previous timestep (scalar or tensor)
        noise_schedule: Noise schedule parameters
        context: Text embeddings (B, seq_len, text_dim) or None
        eta: Stochasticity parameter

    Returns:
        x_prev: Less noisy image
    """
    device = x.device
    alphas_cumprod = noise_schedule['alphas_cumprod']

    # Extract alpha values
    t_index = t[0].item()
    alpha_t = alphas_cumprod[t_index]

    if isinstance(t_prev, torch.Tensor):
        if t_prev.item() >= 0:
            alpha_prev = alphas_cumprod[t_prev.item()]
        else:
            alpha_prev = torch.tensor(1.0, device=device)
    else:
        if t_prev >= 0:
            alpha_prev = alphas_cumprod[t_prev]
        else:
            alpha_prev = torch.tensor(1.0, device=device)

    # Reshape for broadcasting
    alpha_t = alpha_t.view(1, 1, 1, 1)
    alpha_prev = alpha_prev.view(1, 1, 1, 1)

    # Predict noise (with text conditioning)
    predicted_noise = unet(x, t, context=context)

    # Predict x0
    x0_pred = (x - torch.sqrt(1 - alpha_t) * predicted_noise) / torch.sqrt(alpha_t)
    x0_pred = torch.clamp(x0_pred, -1.0, 1.0)

    # Compute variance
    sigma_t = eta * torch.sqrt((1 - alpha_prev) / (1 - alpha_t)) * torch.sqrt(1 - alpha_t / alpha_prev)

    # Compute x_{t-1}
    noise = torch.randn_like(x) if eta > 0 else 0
    x_prev = torch.sqrt(alpha_prev) * x0_pred + \
             torch.sqrt(1 - alpha_prev - sigma_t**2) * predicted_noise + \
             sigma_t * noise

    return x_prev


@torch.no_grad()
def ddim_sample(unet, shape, noise_schedule, device,
                context=None, cfg_scale=1.0,
                ddim_steps=50, eta=0.0):
    """
    DDIM sampling with Classifier-Free Guidance.

    Args:
        unet: UNet model
        shape: Output shape (B, C, H, W)
        noise_schedule: Noise schedule
        device: Device
        context: Text embeddings (B, seq_len, text_dim) or None
        cfg_scale: Guidance scale (1.0 = no guidance)
        ddim_steps: Number of sampling steps
        eta: Stochasticity

    Returns:
        Generated images (B, C, H, W)
    """
    batch_size = shape[0]
    total_timesteps = noise_schedule['timesteps']

    # Create timestep sequence
    c = total_timesteps // ddim_steps
    ddim_timesteps = torch.arange(0, total_timesteps, c, device=device)
    ddim_timesteps = torch.cat([ddim_timesteps, torch.tensor([total_timesteps - 1], device=device)])

    # Start from noise
    x = torch.randn(shape, device=device)

    # Classifier-Free Guidance setup
    use_cfg = (context is not None) and (cfg_scale > 1.0)

    if use_cfg:
        # Create unconditional context (null text)
        uncond_context = torch.zeros_like(context)

    # Reverse process
    for i in reversed(range(len(ddim_timesteps))):
        t = ddim_timesteps[i]
        t_prev = ddim_timesteps[i - 1] if i > 0 else torch.tensor(-1, device=device)
        t_batch = t.repeat(batch_size)

        if use_cfg:
            # Classifier-Free Guidance
            # Predict with text
            noise_pred_cond = unet(x, t_batch, context=context)

            # Predict without text
            noise_pred_uncond = unet(x, t_batch, context=uncond_context)

            # Combine predictions
            noise_pred = noise_pred_uncond + cfg_scale * (noise_pred_cond - noise_pred_uncond)

            # Manual DDIM step with combined prediction
            alpha_t = noise_schedule['alphas_cumprod'][t.item()].view(1, 1, 1, 1)

            if isinstance(t_prev, torch.Tensor):
                if t_prev.item() >= 0:
                    alpha_prev = noise_schedule['alphas_cumprod'][t_prev.item()].view(1, 1, 1, 1)
                else:
                    alpha_prev = torch.tensor(1.0, device=device).view(1, 1, 1, 1)
            else:
                if t_prev >= 0:
                    alpha_prev = noise_schedule['alphas_cumprod'][t_prev].view(1, 1, 1, 1)
                else:
                    alpha_prev = torch.tensor(1.0, device=device).view(1, 1, 1, 1)

            x0_pred = (x - torch.sqrt(1 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)
            x0_pred = torch.clamp(x0_pred, -1.0, 1.0)

            sigma_t = eta * torch.sqrt((1 - alpha_prev) / (1 - alpha_t)) * torch.sqrt(1 - alpha_t / alpha_prev)
            noise = torch.randn_like(x) if eta > 0 else 0
            x = torch.sqrt(alpha_prev) * x0_pred + \
                torch.sqrt(1 - alpha_prev - sigma_t**2) * noise_pred + \
                sigma_t * noise
        else:
            # Standard DDIM (no guidance)
            x = ddim_sample_step(unet, x, t_batch, t_prev, noise_schedule, context, eta)

    return torch.clamp(x, -1.0, 1.0)

In [9]:
# 8.2 Quick Validation Sampling
def quick_validation_sample(unet, text_encoder, tokenizer, noise_schedule, device, epoch, save_dir):
    """
    Generate samples for quick validation during training.
    Uses fixed test prompts.
    """
    print(f"Validation sampling at epoch {epoch}...")

    unet.eval()

    # Test prompts
    test_prompts = [
        # Prompt 1: simple - male human
        "male human with red cape, 4-view sprite sheet",

        # # Prompt 2: medium - female elf
        "female elf with purple skin and blue hair, wearing gold armor, 4-view sprite sheet",

        # Prompt 3: complex = male human + hairstyle
        "male human with dark skin, wearing blue cape, metal gloves and leather boots, 4-view sprite sheet",

        # Prompt 4: complete - female human + multiple equipment
        "lpc-style pixel art character, female human with tanned skin, wearing green hood, cloth pants and metal boots, with blonde hair, 4-view sprite sheet"
    ]

    # Encode prompts
    text_inputs = tokenizer(
        test_prompts,
        padding="max_length",
        max_length=CONFIG["max_text_length"],
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        text_embeddings = text_encoder(**text_inputs).last_hidden_state

    # Generate samples
    samples = ddim_sample(
        unet=unet,
        shape=(len(test_prompts), 4, 128, 128),
        noise_schedule=noise_schedule,
        device=device,
        context=text_embeddings,
        cfg_scale=CONFIG.get("cfg_scale", 7.5),
        ddim_steps=CONFIG.get("default_ddim_steps", 10),
        eta=0.0
    )

    # Save with prompts as filenames
    epoch_dir = Path(save_dir) / f"epoch_{epoch}"
    save_generated_images_with_prompts(samples, test_prompts, epoch_dir, f"epoch{epoch}")

    print(f"Samples saved: {epoch_dir}")

    unet.train()
    return samples

In [10]:
# 8.3 Save Generated Images
def save_generated_images(images, save_dir, filename_prefix="sample"):
    """Save generated images to disk."""
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # Convert from [-1, 1] to [0, 1]
    images = (images + 1.0) / 2.0
    images = torch.clamp(images, 0.0, 1.0)

    # Save individual images
    for i, img in enumerate(images):
        filepath = save_dir / f"{filename_prefix}_{i:04d}.png"
        save_image(img, filepath)

    # Save grid
    if len(images) > 1:
        grid_img = make_grid(images, nrow=2, padding=2, pad_value=1.0)
        grid_path = save_dir / f"{filename_prefix}_grid.png"
        save_image(grid_img, grid_path)

    print(f"  Saved {len(images)} images")


def save_generated_images_with_prompts(images, prompts, save_dir, filename_prefix="sample"):
    """
    Save generated images with prompts.

    Args:
        images: Generated images (B, C, H, W)
        prompts: List of text prompts
        save_dir: Save directory
        filename_prefix: Filename prefix
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # Convert from [-1, 1] to [0, 1]
    images = (images + 1.0) / 2.0
    images = torch.clamp(images, 0.0, 1.0)

    # Save individual images with prompts
    for i, (img, prompt) in enumerate(zip(images, prompts)):
        # Sanitize prompt for filename
        safe_prompt = "".join(c if c.isalnum() or c in (' ', '-', '_') else '_' for c in prompt)
        safe_prompt = safe_prompt[:50]  # Limit length

        filepath = save_dir / f"{filename_prefix}_{i:04d}_{safe_prompt}.png"
        save_image(img, filepath)

    # Save grid
    if len(images) > 1:
        grid_img = make_grid(images, nrow=2, padding=2, pad_value=1.0)
        grid_path = save_dir / f"{filename_prefix}_grid.png"
        save_image(grid_img, grid_path)

    # Save prompts to text file
    prompts_file = save_dir / f"{filename_prefix}_prompts.txt"
    with open(prompts_file, 'w', encoding='utf-8') as f:
        for i, prompt in enumerate(prompts):
            f.write(f"{i}: {prompt}\n")

    print(f"  Saved {len(images)} images with prompts")

## Section 9. Training Loop

In [11]:
# 9.1 Training Loop with Text Conditioning
def train_one_epoch(epoch, unet, text_encoder, tokenizer, train_loader,
                    optimizer, scheduler, scaler, noise_schedule, device, config):
    """
    Train for one epoch with text conditioning.
    Implements Classifier-Free Guidance training.
    """
    unet.train()
    text_encoder.eval()  # CLIP stays frozen

    epoch_loss = 0.0
    num_batches = 0

    # Progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")

    for batch_idx, batch in enumerate(pbar):
        start_time = time.time()

        # Get images and captions
        images = batch['image'].to(device)
        captions = batch['caption']
        batch_size = images.shape[0]

        # Encode text with CLIP
        text_inputs = tokenizer(
            captions,
            padding="max_length",
            max_length=config["max_text_length"],
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            text_embeddings = text_encoder(**text_inputs).last_hidden_state

        # Classifier-Free Guidance: randomly drop text conditioning
        cfg_drop_prob = config.get("cfg_drop_prob", 0.1)
        drop_mask = torch.rand(batch_size, device=device) < cfg_drop_prob

        if drop_mask.any():
            # Create zero embeddings for dropped samples
            text_embeddings = text_embeddings.clone()
            text_embeddings[drop_mask] = 0

        # Random timesteps
        t = torch.randint(0, config["timesteps"], (batch_size,), device=device).long()

        # Add noise
        noise = torch.randn_like(images)
        noisy_images = q_sample(images, t, noise, noise_schedule)

        # Predict noise (with text conditioning)
        if config["use_mixed_precision"]:
            with autocast():
                noise_pred = unet(noisy_images, t, context=text_embeddings)
                loss = F.mse_loss(noise_pred, noise)
        else:
            noise_pred = unet(noisy_images, t, context=text_embeddings)
            loss = F.mse_loss(noise_pred, noise)

        # Backward
        optimizer.zero_grad()

        if config["use_mixed_precision"]:
            scaler.scale(loss).backward()

            if config["gradient_clip"] > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(unet.parameters(), config["gradient_clip"])

            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()

            if config["gradient_clip"] > 0:
                torch.nn.utils.clip_grad_norm_(unet.parameters(), config["gradient_clip"])

            optimizer.step()

        # Update learning rate
        if scheduler is not None:
            scheduler.step()

        # Track loss
        epoch_loss += loss.item()
        num_batches += 1

        # Update progress bar
        current_lr = optimizer.param_groups[0]['lr']
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg': f'{epoch_loss/num_batches:.4f}',
            'lr': f'{current_lr:.6f}'
        })

        # Periodic logging
        if batch_idx % config["log_every"] == 0:
            batch_time = time.time() - start_time
            print(f"\n[Epoch {epoch+1}] [Batch {batch_idx}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}, Avg: {epoch_loss/num_batches:.4f}, "
                  f"LR: {current_lr:.6f}, Time: {batch_time:.2f}s")

    return epoch_loss / num_batches

In [12]:
# 9.2 Checkpoint Management
def save_checkpoint(epoch, unet, optimizer, scheduler, loss, checkpoint_dir,
                   loss_history=None, best_loss=None, keep_last_n=3):
    """Save training checkpoint."""
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(exist_ok=True, parents=True)

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': unet.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'loss': loss,
        'loss_history': loss_history,
        'config': CONFIG,
    }

    # Save epoch checkpoint
    checkpoint_path = checkpoint_dir / f"checkpoint_epoch_{epoch+1}.pt"
    torch.save(checkpoint, checkpoint_path)

    # Update latest
    latest_path = checkpoint_dir / "checkpoint_latest.pt"
    torch.save(checkpoint, latest_path)

    # Check if best
    is_best = False
    if best_loss is None or loss < best_loss:
        best_path = checkpoint_dir / "checkpoint_best.pt"
        torch.save(checkpoint, best_path)

        model_only_path = checkpoint_dir / "model_best.pt"
        torch.save(unet.state_dict(), model_only_path)

        print(f"New best model: {loss:.4f}")
        is_best = True
        best_loss = loss

    # Cleanup old checkpoints
    all_checkpoints = sorted(checkpoint_dir.glob("checkpoint_epoch_*.pt"))
    if len(all_checkpoints) > keep_last_n:
        for old_checkpoint in all_checkpoints[:-keep_last_n]:
            old_checkpoint.unlink()

    return best_loss


def load_checkpoint(checkpoint_path, unet, optimizer=None, scheduler=None, device='cuda'):
    """Load checkpoint and restore training state."""
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    unet.load_state_dict(checkpoint['model_state_dict'])

    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if scheduler is not None and checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    loss_history = checkpoint.get('loss_history', [])
    best_loss = min(loss_history) if loss_history else loss

    print(f"Checkpoint loaded: {Path(checkpoint_path).name}")
    print(f"  Epoch: {epoch+1}")
    print(f"  Loss: {loss:.4f}")
    print(f"  Best loss: {best_loss:.4f}")
    print(f"  History: {len(loss_history)} epochs")

    return epoch, loss, loss_history, best_loss


def save_loss_history(loss_history, save_path):
    """Save loss history to CSV."""
    import csv
    from datetime import datetime

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'loss', 'timestamp'])

        for epoch, loss in enumerate(loss_history, start=1):
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            writer.writerow([epoch, f'{loss:.6f}', timestamp])

    print(f"Loss history saved: {save_path.name}")

In [13]:
# 9.3 Main Training Function
def train(unet, text_encoder, tokenizer, train_loader, optimizer, scheduler, scaler,
          noise_schedule, device, config, start_epoch=0):
    """Main training loop with text conditioning."""
    print(f"\nStarting training:")
    print(f"  Model: Text-Conditional UNet")
    print(f"  Text encoder: {config['clip_model_name']}")
    print(f"  CFG drop prob: {config.get('cfg_drop_prob', 0.1)}")
    print(f"  Epochs: {config['num_epochs']}")
    print(f"  Batches/epoch: {len(train_loader)}")
    print(f"  Starting from: epoch {start_epoch+1}\n")

    loss_history = []
    best_loss = None

    try:
        for epoch in range(start_epoch, config["num_epochs"]):
            epoch_start = time.time()

            # Train one epoch
            avg_loss = train_one_epoch(
                epoch, unet, text_encoder, tokenizer, train_loader,
                optimizer, scheduler, scaler, noise_schedule, device, config
            )

            loss_history.append(avg_loss)
            epoch_time = time.time() - epoch_start

            print(f"\nEpoch {epoch+1} summary: Loss={avg_loss:.4f}, Time={epoch_time/60:.1f}min")

            # Save checkpoint
            if (epoch + 1) % config["save_every_epochs"] == 0:
                best_loss = save_checkpoint(
                    epoch, unet, optimizer, scheduler, avg_loss,
                    checkpoint_dir, loss_history, best_loss,
                    keep_last_n=config.get("keep_last_n", 3)
                )

            # Quick validation
            if (epoch + 1) % config.get("sample_every_epochs", 5) == 0:
                quick_validation_sample(
                    unet, text_encoder, tokenizer, noise_schedule,
                    device, epoch+1, sample_dir
                )

            # Save loss CSV
            loss_csv_path = log_dir / "loss_history.csv"
            save_loss_history(loss_history, loss_csv_path)

    except KeyboardInterrupt:
        print("\nTraining interrupted")
        best_loss = save_checkpoint(
            epoch, unet, optimizer, scheduler, avg_loss,
            checkpoint_dir, loss_history, best_loss,
            keep_last_n=config.get("keep_last_n", 3)
        )
        loss_csv_path = log_dir / "loss_history.csv"
        save_loss_history(loss_history, loss_csv_path)

    # Final checkpoint
    best_loss = save_checkpoint(
        config["num_epochs"]-1, unet, optimizer, scheduler,
        loss_history[-1] if loss_history else 0.0,
        checkpoint_dir, loss_history, best_loss,
        keep_last_n=config.get("keep_last_n", 3)
    )

    loss_csv_path = log_dir / "loss_history.csv"
    save_loss_history(loss_history, loss_csv_path)

    print(f"\nTraining complete:")
    print(f"  Final loss: {loss_history[-1]:.4f}")
    print(f"  Best loss: {best_loss:.4f}")
    print(f"  CSV: {loss_csv_path}")

    return loss_history

In [14]:
# 9.4 Start Training
START_TRAINING = CONFIG.get("start_training", False)
RESUME_TRAINING = CONFIG.get("resume_training", False)

if START_TRAINING and not RESUME_TRAINING:
    # Fresh training
    print("Starting fresh training...")

    loss_history = train(
        unet, text_encoder, tokenizer, train_loader,
        optimizer, scheduler, scaler, noise_schedule,
        device, CONFIG, start_epoch=0
    )

    print("Training completed")

elif RESUME_TRAINING:
    # Resume training
    print("Resuming from checkpoint...")

    checkpoint_path = checkpoint_dir / "checkpoint_latest.pt"

    if checkpoint_path.exists():
        start_epoch, last_loss, previous_loss_history, best_loss = load_checkpoint(
            checkpoint_path, unet, optimizer, scheduler, device
        )

        print(f"Resuming from epoch {start_epoch+1}")

        new_loss_history = train(
            unet, text_encoder, tokenizer, train_loader,
            optimizer, scheduler, scaler, noise_schedule,
            device, CONFIG, start_epoch=start_epoch+1
        )

        combined_loss_history = previous_loss_history + new_loss_history
        combined_csv_path = log_dir / "loss_history_combined.csv"
        save_loss_history(combined_loss_history, combined_csv_path)

        print("Resume training completed")
    else:
        print(f"Checkpoint not found: {checkpoint_path}")

else:
    print("Training not started")
    print("Set CONFIG['start_training'] = True to begin")

print("\nSection 9 complete")

Starting fresh training...

Starting training:
  Model: Text-Conditional UNet
  Text encoder: openai/clip-vit-base-patch32
  CFG drop prob: 0.1
  Epochs: 30
  Batches/epoch: 1041
  Starting from: epoch 1



Epoch 1/30:   0%|          | 0/1041 [00:00<?, ?it/s]

/tmp/ipython-input-1169075543.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



[Epoch 1] [Batch 0/1041] Loss: 1.5279, Avg: 1.5279, LR: 0.000003, Time: 4.04s

[Epoch 1] [Batch 50/1041] Loss: 0.2834, Avg: 0.8108, LR: 0.000100, Time: 0.78s

[Epoch 1] [Batch 100/1041] Loss: 0.1152, Avg: 0.5002, LR: 0.000100, Time: 0.78s

[Epoch 1] [Batch 150/1041] Loss: 0.0786, Avg: 0.3663, LR: 0.000100, Time: 0.78s

[Epoch 1] [Batch 200/1041] Loss: 0.1050, Avg: 0.2942, LR: 0.000100, Time: 0.78s

[Epoch 1] [Batch 250/1041] Loss: 0.0775, Avg: 0.2492, LR: 0.000100, Time: 0.78s

[Epoch 1] [Batch 300/1041] Loss: 0.0896, Avg: 0.2177, LR: 0.000100, Time: 0.78s

[Epoch 1] [Batch 350/1041] Loss: 0.0475, Avg: 0.1945, LR: 0.000100, Time: 0.78s

[Epoch 1] [Batch 400/1041] Loss: 0.0429, Avg: 0.1764, LR: 0.000100, Time: 0.78s

[Epoch 1] [Batch 450/1041] Loss: 0.0447, Avg: 0.1620, LR: 0.000100, Time: 0.78s

[Epoch 1] [Batch 500/1041] Loss: 0.0509, Avg: 0.1506, LR: 0.000100, Time: 0.78s

[Epoch 1] [Batch 550/1041] Loss: 0.0475, Avg: 0.1410, LR: 0.000100, Time: 0.78s

[Epoch 1] [Batch 600/1041] Los

Epoch 2/30:   0%|          | 0/1041 [00:00<?, ?it/s]

...


[Epoch 28] [Batch 0/1041] Loss: 0.0018, Avg: 0.0018, LR: 0.000012, Time: 0.79s

[Epoch 28] [Batch 50/1041] Loss: 0.0025, Avg: 0.0031, LR: 0.000012, Time: 0.78s

[Epoch 28] [Batch 100/1041] Loss: 0.0032, Avg: 0.0029, LR: 0.000012, Time: 0.78s

[Epoch 28] [Batch 150/1041] Loss: 0.0026, Avg: 0.0030, LR: 0.000012, Time: 0.78s

[Epoch 28] [Batch 200/1041] Loss: 0.0036, Avg: 0.0030, LR: 0.000012, Time: 0.78s

[Epoch 28] [Batch 250/1041] Loss: 0.0047, Avg: 0.0029, LR: 0.000012, Time: 0.78s

[Epoch 28] [Batch 300/1041] Loss: 0.0016, Avg: 0.0030, LR: 0.000012, Time: 0.78s

[Epoch 28] [Batch 350/1041] Loss: 0.0027, Avg: 0.0029, LR: 0.000012, Time: 0.78s

[Epoch 28] [Batch 400/1041] Loss: 0.0014, Avg: 0.0029, LR: 0.000012, Time: 0.78s

[Epoch 28] [Batch 450/1041] Loss: 0.0012, Avg: 0.0029, LR: 0.000012, Time: 0.78s

[Epoch 28] [Batch 500/1041] Loss: 0.0030, Avg: 0.0029, LR: 0.000012, Time: 0.78s

[Epoch 28] [Batch 550/1041] Loss: 0.0031, Avg: 0.0030, LR: 0.000012, Time: 0.78s

[Epoch 28] [Batch 

Epoch 29/30:   0%|          | 0/1041 [00:00<?, ?it/s]


[Epoch 29] [Batch 0/1041] Loss: 0.0030, Avg: 0.0030, LR: 0.000011, Time: 0.79s

[Epoch 29] [Batch 50/1041] Loss: 0.0039, Avg: 0.0030, LR: 0.000011, Time: 0.78s

[Epoch 29] [Batch 100/1041] Loss: 0.0040, Avg: 0.0030, LR: 0.000011, Time: 0.78s

[Epoch 29] [Batch 150/1041] Loss: 0.0078, Avg: 0.0030, LR: 0.000011, Time: 0.78s

[Epoch 29] [Batch 200/1041] Loss: 0.0040, Avg: 0.0032, LR: 0.000011, Time: 0.78s

[Epoch 29] [Batch 250/1041] Loss: 0.0039, Avg: 0.0032, LR: 0.000011, Time: 0.78s

[Epoch 29] [Batch 300/1041] Loss: 0.0017, Avg: 0.0033, LR: 0.000011, Time: 0.78s

[Epoch 29] [Batch 350/1041] Loss: 0.0018, Avg: 0.0033, LR: 0.000011, Time: 0.78s

[Epoch 29] [Batch 400/1041] Loss: 0.0037, Avg: 0.0033, LR: 0.000011, Time: 0.78s

[Epoch 29] [Batch 450/1041] Loss: 0.0029, Avg: 0.0032, LR: 0.000011, Time: 0.78s

[Epoch 29] [Batch 500/1041] Loss: 0.0038, Avg: 0.0032, LR: 0.000011, Time: 0.78s

[Epoch 29] [Batch 550/1041] Loss: 0.0013, Avg: 0.0032, LR: 0.000011, Time: 0.78s

[Epoch 29] [Batch 

Epoch 30/30:   0%|          | 0/1041 [00:00<?, ?it/s]


[Epoch 30] [Batch 0/1041] Loss: 0.0022, Avg: 0.0022, LR: 0.000010, Time: 0.79s

[Epoch 30] [Batch 50/1041] Loss: 0.0043, Avg: 0.0031, LR: 0.000010, Time: 0.78s

[Epoch 30] [Batch 100/1041] Loss: 0.0033, Avg: 0.0030, LR: 0.000010, Time: 0.78s

[Epoch 30] [Batch 150/1041] Loss: 0.0034, Avg: 0.0032, LR: 0.000010, Time: 0.78s

[Epoch 30] [Batch 200/1041] Loss: 0.0022, Avg: 0.0031, LR: 0.000010, Time: 0.78s

[Epoch 30] [Batch 250/1041] Loss: 0.0039, Avg: 0.0031, LR: 0.000010, Time: 0.78s

[Epoch 30] [Batch 300/1041] Loss: 0.0055, Avg: 0.0030, LR: 0.000010, Time: 0.78s

[Epoch 30] [Batch 350/1041] Loss: 0.0010, Avg: 0.0030, LR: 0.000010, Time: 0.78s

[Epoch 30] [Batch 400/1041] Loss: 0.0038, Avg: 0.0030, LR: 0.000010, Time: 0.78s

[Epoch 30] [Batch 450/1041] Loss: 0.0008, Avg: 0.0030, LR: 0.000010, Time: 0.78s

[Epoch 30] [Batch 500/1041] Loss: 0.0045, Avg: 0.0030, LR: 0.000010, Time: 0.78s

[Epoch 30] [Batch 550/1041] Loss: 0.0049, Avg: 0.0030, LR: 0.000010, Time: 0.78s

[Epoch 30] [Batch 